After we trained the model, how do we evaluate and quantify its performance?
## Partitioning the dataset


When we want to evaluate a model, we need to test it on data that it has never seen before. This is important to assess how well the model generalizes to new, unseen data.
To do this, we typically partition the dataset into two or more subsets:
- **Training set**: This subset is used to train the model. The model learns the patterns and relationships in the training data.
- **Test set**: This subset is used to evaluate the model's performance. The model makes predictions on the test data, and we compare these predictions to the actual labels to assess accuracy.



There are different strategies to partition the dataset:
### 1. Holdout method
The dataset is randomly split into a training set and a test set, typically using a ratio such as 80% for training and 20% for testing. 

The ratio can vary depending on the size of the dataset and the specific requirements of the analysis: for a large dataset, a "smaller" training set  may be sufficient, while for a small dataset, a larger training set may be necessary to train the model effectively.

This is because if we don't have enough data in the training set, the model may not learn the underlying patterns and relationships effectively, leading to poor performance on unseen data.  
This is clearer when we look at the learning curve: the curve shows how the accuracy of the model changes as the size of the training set (sample size) increases. With a small training set, the variance of the accuracy is high, meaning that the model's performance can vary significantly depending on the specific data points included in the training set. As the training set size increases, the variance decreases, and the accuracy stabilizes, indicating that the model is learning more effectively from a larger and more representative sample of the data.

<img src="img_teoria/learningcurve.png" width="500"/>

*Remember that once the model is trained on the training set, tested and we decided the hyperparameters, we should **train it again on the entire dataset (training + test) before deploying it in production**, to make use of all available data for training.*

**Hold out method is good for large datasets.** 
(for small datasets as seen in the graph it can lead to high variance in the evaluation metrics, since the performance can depend heavily on which data points end up in the training and test sets. In such cases, cross-validation is often preferred)

### 2. K-Fold Cross-Validation
Imagine to have a dataset with limited samples, i.e. only 100 instances. If we use the holdout method with an 80-20 split, we would have only 80 instances for training and 20 for testing. This small training set may not be sufficient for the model to learn effectively, leading to poor performance on unseen data.  

To address this issue, we can use k-fold cross-validation. In this method, the dataset is divided into k equal-sized folds (subsets). The model is trained and evaluated k times, each time using a different fold as the test set and the remaining (k-1) folds as the training set. This way, every instance in the dataset is used for both training and testing, providing a more robust evaluation of the model's performance.

After all k iterations, the performance metrics (e.g., accuracy, precision, recall) are averaged to obtain a final estimate of the model's performance.

![](img_teoria/k_fold.png)

Of course this method is used only for small datasets, since it requires training the model k times, which can be computationally expensive for large datasets.

The worst case, for very small datasets, is the **Leave-One-Out** Cross-Validation, where *k is equal to the number of instances in the dataset*. In this case, each instance is used as a test set once, and the model is trained on all other instances. This method provides an almost unbiased estimate of the model's performance, but it can be very computationally expensive for large datasets.

## Model Performance Estimation
Talking about partitioning the dataset, that was the simple pipeline, where we already decided the model and its hyperparameters. But how do we choose the best model and hyperparameters in the first place?

In reality, the process is a bit more complex, and usually involves three steps: **Training**, **Model Validation**, and **Testing**:
- **Training**: The model is trained on the training set.
- **Model Validation**: Different models and hyperparameters are evaluated on a validation set to select the best one. Actually, the other models are of course trained on the training set as well, but their performance is evaluated on the validation set, from which we can tune the hyperparameters (ex. for decision trees depht, for forests number of trees, for k value of k and different distance metrics...) and select the best model. 
- **Testing**: The final selected model is evaluated on the test set to estimate its performance on unseen data.

So the validation set is an additional subset of the data used to tune the model and select the best hyperparameters. If we didn't have it and only rely on the test set, we would risk overfitting the model to the test data, leading to an overly optimistic estimate of its performance (we would not only train the model on the training set, but also choose the best hyperparameters based on the test set performance, which is not a good practice, overfitting to the test set!).

Typically, using Hold-Out (if the dataset is big), the dataset is split into three parts: 60% for training, 20% for validation, and 20% for testing. However, these ratios can vary depending on the size of the dataset and the specific requirements of the analysis as seen before.

For cross-validation, if the dataset is small, the idea is to use hold-out to split in training+validation and test set, and then use k-fold cross-validation on the training+validation set to select the best model and hyperparameters (here k-1 folds for training and 1 fold for validation, changing the validation fold at each iteration). Finally, the selected model (i.e. the one with the best performance on the validation set) is evaluated on the test set.

In definitiva, per cross validation, si parte dal dataset che viene diviso in due parti: la prima per il test e la seconda training+validation. Poi con k folds: si divide training+validation in k parti uguali, per k volte si fa il training su k-1 e validation sull'ultimo e dopo queste k iterazioni si calcola la media dell'evaluation per capire quanto è buono il modello. Questo procedimento si fa per tutte le scelte di iperparametri/modelli e alla fine si sceglie quello con la performance migliore.

## Model Validation Strategies
How to find a "good choice" of hyperparameters for a model? 

There are different strategies for model validation:
### Manual Search
It is the simplest method: we manually select a set of hyperparameter values to try, train the model with each combination on the training set, and evaluate its performance on the validation set. We then choose the combination that yields the best performance.
### Grid Search


We define a set of relevant hyperparameters and their possible values (for example, in KNN, the relevant parameters could be the number of neighbors k and the distance metric). We then create a grid of all possible combinations of these hyperparameter values (i.e. we compute the Cartesian Product). The model is then trained (on the training set) and evaluated (on the validation set) for each combination, and the one that yields the best performance is selected.  
Below an example of grid search for KNN with different values of k and distance metrics, the one in the center indicates that the best performance was achieved with k=3 and L1 distance metric.

<img src="img_teoria/grid_search_1.png" width="150"/>

Grid Search garantees to find the optimal combination of hyperparameters within the defined grid, but it can be computationally expensive, especially when the number of hyperparameters and their possible values is large.  
Fortunately it is parallelizable, since each combination is independent from the others, so we can train and evaluate multiple models simultaneously on different processors or machines. Anyway it is still computationally expensive, so a more efficient alternative is Random Search.

### Random Search
Instead of evaluating all possible combinations of hyperparameters, we randomly sample a fixed number of combinations from the grid (this fixed number is our "budget"), in order to limit the number of models to train and evaluate (remember that each combination requires in fact training and evaluating a model). 

This approach allows us to explore a larger hyperparameter space with a limited computational budget, so it is not computationally expensive as Grid Search. Anyway, since it is based on random sampling, there is no guarantee that we will find the optimal combination of hyperparameters, but in practice it often yields good results with much less computational cost compared to Grid Search.

Also Random Search can be parallelized, since each sampled combination is independent from the others, allowing for simultaneous training and evaluation of multiple models on different processors or machines. So Random Search doesn't learn from previous evaluations...

<img src="img_teoria/grid_search_2.png" width="150"/>

### Bayesian Optimization
Bayesian Optimization is a strategy used to find the maximum of an unknown (black box) function that is expensive to evaluate. In the context of hyperparameter tuning, the unknown function is the model's performance (e.g., accuracy) as a function of its hyperparameters.

Instead of randomly sampling combinations or exhaustively searching through a grid, Bayesian Optimization builds a surrogate model (often a Gaussian Process) to approximate the relationship between hyperparameters and model performance.  
So, for each iteration, the algorithm selects the next set of hyperparameters to evaluate by balancing **exploration** (trying new areas of the hyperparameter space) and **exploitation** (focusing on areas known to yield good performance).

Let's see an example: we start without knowing anything about the function, so we try a random point (hyperparameter combination) and evaluate the function (train the model and evaluate its performance). Based on this evaluation, we update our surrogate model with the new information. In the example we tried k=2 and cosine distance, and we obtained a certain accuracy (let's say 70%). The surrogate model is updated to reflect this new data point: we could assume to have similar performance for nearby hyperparameter values (like k=2 and cosine distance or in general using L1 distance), but can't tell anything about far away points. So, following the principle of exploration vs exploitation, we try a new point that is far away from the previous one (like k=5 and L2 distance), and we evaluate the function again, obtaining a new accuracy (in the example we get a bad accuracy) and so updating the surrogate model. This process continues iteratively, with the surrogate model becoming more accurate over time, allowing us to make more informed decisions about which hyperparameter combinations to evaluate next, stopping once the budget is exhausted (i.e., we have evaluated a fixed number of combinations).

<p align="center">
  <img src="img_teoria/bayesian_1.png" width="30%">
  <img src="img_teoria/bayesian_2.png" width="30%">
  <img src="img_teoria/bayesian_3.png" width="30%">
</p>


The Bayesian Optimization is more efficient than Grid Search and Random Search, as it uses the information from previous evaluations to guide the search for optimal hyperparameters. 

Anyway it is not easily parallelizable, since each evaluation depends on the results of previous evaluations, and also the use of a surrogate model adds computational overhead.

("the use of a surrogate model adds computational overhead" means that building and updating the surrogate model (like a Gaussian Process) requires additional computations compared to simply evaluating the model's performance for a given set of hyperparameters. This overhead can be significant, especially if the surrogate model is complex or if the number of hyperparameters is large).

## Metrics for Model Evaluation
To evaluate the performance of a model, we use various metrics depending on the type of problem (classification, regression, etc.). 

Talking about classification, we first introduce the concept of **Confusion Matrix**.  
The confusion matrix is a table with number of rows and columns equal to the number of classes in the classification problem. Each row of the matrix represents the instances in an actual class, while each column represents the instances in a predicted class (or vice versa). The entries in the matrix indicate the number of instances that were classified correctly or incorrectly for each class.

In the example below we have a binary classification problem with two classes: Positive and Negative. The confusion matrix shows the counts of: 
- true positives (TP) if predicted Positive and actually Positive,
- true negatives (TN) if predicted Negative and actually Negative,
- false positives (FP) if predicted Positive but actually Negative,
- false negatives (FN) if predicted Negative but actually Positive.

<img src="img_teoria/confusion_matrix.png" width="500"/>

The most widely used metrics for model evaluation derived from the confusion matrix is Accuracy. Accuracy is defined as the ratio of correctly predicted instances to the total instances in the dataset:

$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$

This metric gives us an overall idea of how well the model is performing, but it may not be sufficient in cases where the classes are imbalanced (i.e., one class has significantly more instances than the other). For example, if 95% of the instances belong to the Negative class, a model that always predicts Negative will have an accuracy of 95%, but it is not a useful model since it fails to identify any Positive instances.

Also classes may have different importance: for example in medical diagnosis, predicting a disease when it is not present (false positive) may be less critical than failing to predict a disease when it is present (false negative). In this case Accuracy is not appropriate because it treats all types of errors equally.

**So Accuracy is not appropriate for unbalanced class label distributions or if classes have different importance.**

In order to address these issues, we can use other metrics derived from the confusion matrix, such as Precision, Recall, and F1-Score. **All these metrics focus on a specific class, usually the Positive in the binary classification case.**
- **Precision** is the ratio of correctly predicted instances of a certain class C to the total predicted instances of class C. Below the formula for Precision for the Positive class:

  $\text{Precision} = \frac{TP}{TP + FP}$

  Precision tells us how many of the instances predicted as Positive are actually Positive. It is useful when the cost of false positives is high (i.e. in order to minimize false positives, we must maximize Precision).

- **Recall** (also known as Sensitivity or True Positive Rate) is the ratio of correctly predicted instances of a certain class C to the total actual instances of class C. Below the formula for Recall for the Positive class:

    $\text{Recall} = \frac{TP}{TP + FN}$

    Recall tells us how many of the actual Positive instances were correctly identified by the model. It is useful when the cost of false negatives is high (i.e. in order to minimize false negatives, we must maximize Recall).

- **F1-Score** is the harmonic mean of Precision and Recall, providing a single metric that balances both Precision and Recall. It is defined as:

  $\text{F1-Score} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$

So Precision and Recall are different concepts: Precision tells us how good the model is at predicting Positive instances (i.e., how many of the predicted Positive instances are actually Positive), while Recall tells us how good the model is at identifying all actual Positive instances (i.e., how many of the actual Positive instances were correctly predicted by the model).

If we want high recall, we could start saying that every instance is Positive, so we would have no false negatives (FN=0) and thus Recall=1. But in this way we would have many false positives (FP), leading to low Precision.  
Conversely, if we want high Precision, we could become very conservative and classify an instance as Positive only when we are almost certain.
This would reduce the number of False Positives (FP ≈ 0), but we would miss many actual positives since we take a positive only if we are almost certain -> FN increases, leading to low Recall.
**So the F1-Score is useful to balance Precision and Recall, especially when we want to find a trade-off between the two metrics.**

## Learning Curve
The Learning Curve shows how the performance of a model (accuracy, in our case) changes as the size of the training set increases. In the image, the bars represent the variance of the accuracy at different training set sizes. 

For small training set sizes, the variance is high, meaning that the model's performance can vary significantly depending on the specific data points included in the training set. As the training set size increases, the variance decreases, and the accuracy stabilizes, indicating that the model is learning more effectively from a larger and more representative sample of the data.

Along the x-axis the size of the training set is in log scale. **The important thing to note is that passing from a sample size of 10 to 100 has a huge impact on both variance and accuracy, while passing from 1,000 to 10,000 has a smaller impact**. This means that increasing excessively the training set size has diminishing returns in terms of accuracy improvement, so often we arrive at a point where adding more data does not significantly improve the model's performance.

<img src="img_teoria/learningcurve.png" width="500"/>

## ROC Curve e AUC
La ROC Curve (Receiver Operating Characteristic) serve per valutare le performance di un modello di classificazione binaria mostrando il trade-off tra True Positive Rate (TPR) e False Positive Rate (FPR) a diversi threshold di classificazione.
- **TPR** (o Recall) è la proporzione di veri positivi correttamente identificati dal modello rispetto a tutti i veri positivi originali, calcolata come TP / (TP + FN).  
*Misura la capacità del modello di identificare correttamente i casi positivi.*
- **FPR** è la proporzione di falsi positivi rispetto a tutti i negativi, calcolata come FP / (FP + TN).  
*Misura la frequenza con cui il modello classifica erroneamente i casi negativi come positivi.*


Nel grafico della ROC Curve, l'asse x rappresenta il FPR, mentre l'asse y rappresenta il TPR. Ogni punto sulla curva corrisponde a un diverso threshold di classificazione: spostandosi lungo la curva, si può osservare come variano TPR e FPR al variare del threshold.

Per capire la questione dei threshold, è necessario anzitutto sapere che molti modelli di classificazione binaria (come la regressione logistica o le reti neurali) non producono direttamente etichette di classe (Positive o Negative), ma piuttosto una probabilità associata alla classe positiva.  
Si può assegnare un **thresholding value $\tau$** per decidere a partire da quale probabilità un'istanza viene classificata come Positive: **If $P(+|\mathbf{x}) \geq \tau$, predict x as Positive. Otherwise, predict as Negative.**

Quindi il modello assegna ad ogni istanza una probabilità di appartenere alla classe Positive, e in base al valore di $\tau$ si decide se classificare l'istanza come Positive o Negative. Nei tre grafici successivi si osservano degli istogrammi. Sull'asse x c'è la probabilità predetta dal modello che l'istanza appartenga alla classe Positive, mentre sull'asse y c'è il numero di istanze con quella probabilità. Le barre verdi rappresentano le istanze effettivamente Positive, mentre le barre rosse rappresentano le istanze effettivamente Negative.

Immaginiamo in questo caso di trattare le istanze del training set.  
Allora maggiormente le barre sono separate (cioè le istanze Positive hanno probabilità alte e le Negative basse), migliore è il modello, perché significa che il modello riesce a distinguere bene tra le due classi e possiamo trovare un threshold ottimale.  
Se il modello non è stato in grado di separare bene le due classi, le barre saranno miste, con molte istanze Positive con probabilità bassa e molte Negative con probabilità alta -> difficile trovare un threshold che ci dia buone performance.

Si vuole scegliere un threshold $\tau$ che massimizzi TPR e che minimizzi FPR. Nella prima immagine, se ci spostiamo a destra aumentando il threshold, riduciamo gli errori di classificazione delle istanze Negative (FPR diminuisce), ma aumentiamo gli errori sulle istanze Positive (TPR diminuisce). Al contrario spostandoci a sinistra aumentando TPR, aumentano anche gli errori sulle istanze Negative (FPR aumenta).  
Si deve quindi trovare un compromesso tra i due obiettivi, scegliendo un threshold che bilancia TPR e FPR in modo adeguato. 

La prima figura mostra un modello con una buona separazione tra le classi Positive e Negative, permettendo di scegliere un threshold che bilancia bene TPR e FPR.  
La seconda figura mostra un modello con una separazione estremamente netta tra le classi, consentendo di ottenere sia TPR che FPR molto bassi scegliendo un threshold appropriato. Tuttavia questo scenario è raro nella pratica e suggerisce overfitting.  
La terza figura mostra un modello con una scarsa separazione tra le classi Positive e Negative, rendendo difficile trovare un threshold che bilanci bene TPR e FPR. In questo caso, il modello non è in grado di distinguere efficacemente tra le due classi, portando a prestazioni inferiori.

<p align="center">
  <img src="img_teoria/isto1.png" width="30%">
  <img src="img_teoria/isto2.png" width="30%">
  <img src="img_teoria/isto3.png" width="30%">
</p>


Possiamo utilizzare la ROC Curve per capire come la scelta del threshold influisce sulla performance del modello in termini di TPR e FPR.  
Ricordiamo che lungo l'asse x c'è il FPR, mentre lungo l'asse y c'è il TPR. Ogni punto sulla curva corrisponde a un diverso threshold di classificazione. Es. nella figura sotto, per la curva rossa, possiamo individuare un punto di threshold per cui si hanno TPR = 0.4 e FPR = 0.1.

In generale, ogni curva ROC inizia dal punto in basso a sinistra (0,0) e termina nel punto in alto a destra (1,1), questo perché:
- Se scegliamo un threshold pari a 0 -> il modello tenderà a classificare tutte le istanze come Positive, portando a un alto TPR ma anche a un alto FPR (nella curva, questo corrisponde al punto in alto a destra (1,1)).
- Se scegliamo un threshold pari a 1 -> il modello tenderà a classificare tutte le istanze come Negative, portando a un basso FPR ma anche a un basso TPR (nella curva, questo corrisponde al punto in basso a sinistra (0,0)).  
L'ideale (utopico) sarebbe avere un modello che raggiunge il punto in alto a sinistra della curva, cioè con TPR=1 e FPR=0. 

Se la linea della ROC Curve **è vicina alla diagonale** (blu) -> per ogni threshold TPR e FPR sono uguali. Questo rappresenta un modello che effettua **random guessing**, non è in grado di distinguere tra le due classi e quindi la sua performance è equivalente a quella di un classificatore casuale.

Se la linea della ROC Curve **è sopra e lontana dalla diagonale** (come la rossa) -> il modello è in grado di distinguere tra le due classi, ottenendo un TPR più alto rispetto al FPR per vari threshold. 

Se la linea della ROC Curve **è sotto la diagonale** -> il modello sta facendo peggio del random guessing. In questo caso, invertendo le previsioni del modello (cioè classificando come Positive le istanze che il modello classifica come Negative e viceversa), si otterrebbe un modello migliore del random guessing.

L'ideale (utopico) sarebbe avere un'area sotto la ROC Curve (AUC - Area Under the Curve) pari a 1, che indica un modello perfetto in grado di distinguere tra le due classi senza errori. Un AUC di 0.5 indica un modello che effettua random guessing, mentre un AUC inferiore a 0.5 indica un modello che fa peggio del random guessing.

<img src="img_teoria/roc.png" width="300"/>

### Costruire la ROC Curve
Per costruire la ROC Curve, si seguono i seguenti passi:
- Si calcolano le probabilità predette dal modello per ogni istanza del dataset (ad esempio, la probabilità che un'istanza appartenga alla classe Positive).
- Si ordinano le istanze in base a queste probabilità, dalla più alta alla più bassa.
- Si calcolano il TPR e il FPR per vari threshold, tracciando i punti corrispondenti sulla curva.

Nell'esempio sotto, si vede come mettendo un threshold pari a 0.43 si ha un TPR di 0.8 (4/5, se prendo threshold 0.43 allora tutte le istanze con probabilità >= 0.43 sono considerate Positive, quindi 4 sono True Positive mentre tutti i negativi sono classificati come positivi quindi FPR = 1) e un FPR di 1. Con un threshold pari a 0.87, si ha un TPR di 0.4 (2/5) e un FPR di 0.2 (1/5).

<img src="img_teoria/roc_example1.png" width="400"/>
<img src="img_teoria/roc_example2.png" width="500"/>

## Usare la ROC curve per confrontare modelli
La ROC Curve può essere utilizzata per confrontare le performance di diversi modelli di classificazione binaria. Un modello con una ROC Curve più vicina al punto in alto a sinistra (TPR=1, FPR=0) è considerato migliore, poiché indica una maggiore capacità di distinguere tra le classi Positive e Negative.

Nell'esempio sotto, abbiamo due curve. La curva M1 (rossa) è più alta di M2 per bassi valori di FPR, ma per FPR più alti la curva M2 (verde) supera M1.  
Questo significa che nessuno dei due modelli è sempre migliore dell'altro: M1 è preferibile se vogliamo mantenere basso l'FPR, mentre M2 è preferibile se siamo disposti ad accettare un FPR più alto in cambio di un TPR più elevato.

<img src="img_teoria/roc_compare.png" width="500"/>